# Chapter 2 - Tokens, Cost and Model Selection
## Hands-On Colab Lab for Network Engineers

**The mystery this notebook solves:**
> Tuesday: analysed 20 large router configs - cost 
> Wednesday: analysed 50 small switch configs - cost 
> *More configs but LESS cost. Why?*

The answer is **tokens** - the fundamental unit of LLM billing. By the end of this notebook you will understand exactly why, and you will have a real **fleet cost projection tool** you can use before any batch AI operation.

| # | Demo | What it teaches |
|---|------|----------------|
| 1 | Setup | Install tiktoken, connect to Claude API |
| 2 | BPE from scratch | WHY GigabitEthernet splits into 3 tokens |
| 3 | Network text tokenisation | Token cost of configs, IP addresses, BGP commands |
| 4 | Numbers are expensive | Why your IP addresses cost more than you think |
| 5 | Multi-model cost calculator | Compare cost across all major models |
| 6 | tiktoken vs Claude API | Estimate vs exact - and how much they differ |
| 7 | The Cascade Pattern | Cheap model first, escalate only when needed |
| 8 | Fleet cost projection tool | Plan batch costs before you spend |

**~25 minutes** | **~\/usr/bin/bash.05 in API calls** | **No local setup required**


---
## Demo 1 - Setup

We need two libraries:
- **anthropic** - to call the Claude API and get exact token counts
- **tiktoken** - OpenAIs open-source tokeniser, a good fast approximation for any BPE model


In [ ]:
!pip install -q anthropic tiktoken
print("Libraries installed")


In [ ]:
# TOKENIZATION BASICS: How the AI breaks text into pieces
# Before the AI can process 'show ip route', it splits it into tokens:
#   'show' → one token, 'ip' → one token, 'route' → one token
# Network terms like 'GigabitEthernet' may split into multiple tokens,
# which means they cost more and the AI might misinterpret them.
import os, json, re
import tiktoken
from anthropic import Anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets")
except Exception:
    os.environ["ANTHROPIC_API_KEY"] = input("Paste your Anthropic API key: ")

client = Anthropic()
MODEL  = "claude-haiku-4-5-20251001"  # Cheapest - perfect for cost demos

enc = tiktoken.get_encoding("cl100k_base")
print(f"tiktoken encoder ready. Vocab size: {enc.n_vocab:,} tokens")


---
## Demo 2 - BPE From Scratch: WHY Does GigabitEthernet Split Into 3 Tokens?

**Byte-Pair Encoding (BPE)** is the algorithm that converts text into tokens.
Understanding it explains why network configs tokenise inefficiently.

The algorithm:
1. Start with individual characters as the vocabulary
2. Count the most frequent adjacent pair of tokens
3. Merge that pair into one new token
4. Repeat until vocabulary reaches target size (~50,000-100,000 tokens)

Common words -> single tokens. Rare technical words -> multiple tokens.

> **Networking analogy:** BPE is like MPLS label assignment.
> Common destinations get short, efficient labels.
> Unusual destinations fall back to longer combinations.


In [ ]:
# TOKEN COUNTING: Estimate cost before sending data to the AI
# Every API call is billed by tokens (input + output).
# A typical 'show run' output = 2000-5000 tokens ≈ $0.002-$0.01
# This helps you budget before sending large config files.
from collections import Counter

def get_vocab(corpus):
    vocab = Counter()
    for word in corpus.lower().split():
        vocab[" ".join(list(word)) + " </w>"] += 1
    return vocab

def get_pairs(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_pair(pair, vocab):
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

networking_corpus = ("interface gigabitethernet interface fastethernet interface loopback\n"
    "router ospf router bgp router eigrp\n"
    "neighbor network area cost metric\n"
    "ip address subnet mask prefix\n"
    "shutdown description no shutdown")

print(f"Corpus: {len(networking_corpus.split())} words")
vocab = get_vocab(networking_corpus)

merge_log = []
for step in range(15):
    pairs = get_pairs(vocab)
    if not pairs: break
    best_pair = pairs.most_common(1)[0][0]
    merged    = "".join(best_pair)
    merge_log.append((step+1, best_pair, merged, pairs[best_pair]))
    vocab = merge_pair(best_pair, vocab)

print(f"{'Step':<6} {'Merged pair':<30} {'New token':<20} {'Frequency'}")
print("-" * 65)
for step, pair, merged, freq in merge_log:
    print(f"{step:<6} {str(pair):<30} {merged:<20} {freq}")

print()
print("Notice: in+terface -> interface happens early (very common).")
print("This is exactly why GigabitEthernet takes more tokens than interface.")


In [ ]:
# BPE (Byte Pair Encoding): The algorithm behind tokenization
# Starts with individual characters, then merges frequent pairs.
# Common words like 'the' become single tokens (cheap).
# Rare words like 'GigabitEthernet0/0/0' become many tokens (expensive).
def show_tokens(text):
    tokens = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"  Input  : {repr(text)}")
    print(f"  Tokens : {decoded}")
    print(f"  Count  : {len(tokens)} tokens")
    print()

print("How a PRODUCTION tokeniser splits networking commands:")
print("=" * 60)

examples = [
    ("interface GigabitEthernet0/0",     "Common word + technical word + numbers"),
    ("interface Loopback0",               "Common word + moderately common word"),
    ("ip address 192.168.10.254",         "Two common words + IP address"),
    ("router ospf 1",                     "Common words + single digit"),
    ("neighbor 10.0.0.1 remote-as 65001", "Complex BGP command"),
    ("no shutdown",                       "Two common words"),
    ("Hello world",                       "Plain English baseline"),
]

for text, note in examples:
    print(f"  [{note}]")
    show_tokens(text)


---
## Demo 3 - Network Text Tokenisation: The Real Cost of Configs

Now that you understand *why* text splits the way it does,
let's measure the real token cost of actual networking content.

The key insight: **network configs tokenise at a WORSE ratio than plain English**
because of IP addresses, interface names, and numbers.


In [ ]:
# VOCABULARY ANALYSIS: How well does the AI know network terms?
# Common English words → single tokens (efficient)
# Network-specific terms → multiple tokens (less efficient, may confuse AI)
# This matters because: more tokens = higher cost + more chance of errors.
def token_report(text, label):
    tokens = enc.encode(text)
    chars  = len(text)
    ratio  = chars / len(tokens) if tokens else 0
    return {"label": label, "chars": chars, "tokens": len(tokens), "ratio": ratio}

samples = [
    ("Hello world, this is a test sentence for English baseline.", "Plain English"),
    ("interface GigabitEthernet0/0\n ip address 192.168.1.1 255.255.255.0\n no shutdown", "IOS interface block"),
    ("router bgp 65001\n neighbor 203.0.113.1 remote-as 1234\n neighbor 203.0.113.1 password MyBGPkey", "BGP config block"),
    ("10.0.0.0/8 10.1.1.0/24 10.2.2.0/24 172.16.0.0/12 192.168.0.0/16", "IP prefixes"),
    ("255.255.255.0 255.255.255.128 255.255.254.0 255.255.252.0", "Subnet masks"),
    ("65001 65002 65100 65200 65535 65001 65002", "BGP AS numbers"),
]

print(f"{'Text type':<25} {'Chars':>7} {'Tokens':>8} {'Chars/token':>12}  Efficiency")
print("-" * 72)

for text, label in samples:
    r = token_report(text, label)
    bar = "#" * int(r["ratio"] * 3)
    efficiency = "Good" if r["ratio"] > 4.5 else ("OK" if r["ratio"] > 3.5 else "Poor")
    print(f"{label:<25} {r['chars']:>7} {r['tokens']:>8} {r['ratio']:>11.1f}  {efficiency}  {bar}")

print()
print("Insight: Plain English ~ 4.5 chars/token (efficient)")
print("         Network configs ~ 3.0-3.5 chars/token (25-35% more tokens than expected)")


In [ ]:

# Sample Cisco IOS router config used throughout this notebook
sample_config = """\
hostname core-router-01
!
! === INTERFACES ===
interface GigabitEthernet0/0
 description Uplink-to-ISP-A-Primary
 ip address 203.0.113.1 255.255.255.252
 ip access-group INBOUND-FILTER in
 no shutdown
!
interface GigabitEthernet0/1
 description Uplink-to-ISP-B-Secondary
 ip address 198.51.100.1 255.255.255.252
 no shutdown
!
interface GigabitEthernet0/2
 description LAN-Core-Segment
 ip address 10.0.0.1 255.255.255.0
 ip helper-address 10.0.1.10
 no shutdown
!
interface Loopback0
 description Router-ID
 ip address 192.168.255.1 255.255.255.255
!
! === ROUTING ===
router ospf 1
 router-id 192.168.255.1
 network 10.0.0.0 0.0.255.255 area 0
 network 192.168.255.1 0.0.0.0 area 0
 passive-interface GigabitEthernet0/0
 passive-interface GigabitEthernet0/1
!
router bgp 65001
 bgp router-id 192.168.255.1
 bgp log-neighbor-changes
 neighbor 203.0.113.2 remote-as 1234
 neighbor 203.0.113.2 description ISP-A-BGP-peer
 neighbor 203.0.113.2 soft-reconfiguration inbound
 neighbor 198.51.100.2 remote-as 5678
 neighbor 198.51.100.2 description ISP-B-BGP-peer
 neighbor 198.51.100.2 soft-reconfiguration inbound
 !
 address-family ipv4
  network 203.0.113.0 mask 255.255.255.0
  neighbor 203.0.113.2 activate
  neighbor 198.51.100.2 activate
 exit-address-family
!
! === ACCESS LISTS ===
ip access-list extended INBOUND-FILTER
 deny   ip 10.0.0.0 0.255.255.255 any
 deny   ip 172.16.0.0 0.15.255.255 any
 deny   ip 192.168.0.0 0.0.255.255 any
 permit ip any any
!
! === NTP / LOGGING ===
ntp server 216.239.35.0
ntp server 216.239.35.4
logging host 10.0.1.20
logging buffered 16384
service timestamps log datetime msec localtime
!
! === SNMP ===
snmp-server community ReadOnlyCommunity RO
snmp-server location Munich-DC1-Rack3
snmp-server contact noc@example.com
!
end
"""

sections = [
    ("Hostname + comments",   "hostname core-router-01\n!"),
    ("One interface block",   "interface GigabitEthernet0/0\n description Uplink-to-ISP-A-Primary\n ip address 203.0.113.1 255.255.255.252\n no shutdown"),
    ("OSPF section",          "router ospf 1\n router-id 192.168.255.1\n network 10.0.0.0 0.0.255.255 area 0"),
    ("BGP section",           "router bgp 65001\n neighbor 203.0.113.2 remote-as 1234"),
    ("Full config",            sample_config),
]

print(f"{'Section':<30} {'Tokens':>8}  {'Haiku cost':>12}  {'Sonnet cost':>13}")
print("-" * 70)

for label, text in sections:
    t = len(enc.encode(text))
    print(f"{label:<30} {t:>8}  ${t*0.8e-6:>11.6f}  ${t*3e-6:>12.6f}")

print()
print("Rule of thumb:  ~500-line config  ~ 2,000-4,000 tokens")
print("                ~5,000-line config ~ 20,000-40,000 tokens")
print("                A large core router (~50K lines) ~ 200,000+ tokens")


---
## Demo 4 - Numbers Are Expensive

This is the most surprising finding for network engineers.
**Every digit in your config is often its own token.**

IP addresses, AS numbers, VLAN IDs, interface numbers, OSPF metrics -
they all tokenise digit-by-digit in many tokenisers.

> **Why it matters:** A BGP table with 900,000 routes would have
> millions of tokens just for the prefix numbers - even before any keywords.


In [ ]:
number_examples = [
    ("1",            "Single digit AS number"),
    ("65001",        "5-digit AS number"),
    ("65535",        "Max private AS number"),
    ("100",          "OSPF cost / VLAN ID"),
    ("4294967295",   "32-bit max value"),
    ("10.0.0.1",     "IP address (small)"),
    ("172.16.254.1", "IP address (medium)"),
    ("203.0.113.254","IP address (documentation)"),
    ("255.255.255.0","Subnet mask"),
    ("0.0.0.255",    "Wildcard mask"),
]

print(f"{'Value':<20} {'What it is':<30} {'Tokens':>7}  Token breakdown")
print("-" * 85)

for val, label in number_examples:
    tokens  = enc.encode(val)
    decoded = [repr(enc.decode([t])) for t in tokens]
    print(f"{val:<20} {label:<30} {len(tokens):>7}  {decoded}")

print()
print("Key insight:")
print("  Small common numbers (1, 10, 100) -> 1 token")
print("  Large numbers (65001, 65535)      -> 2-3 tokens")
print("  IP addresses                      -> 5-9 tokens")
print("  Subnet masks                      -> 7-9 tokens")


In [ ]:
strategies = [
    ("Full interface name",  "interface GigabitEthernet0/0",          None),
    ("Abbreviated int name", "int Gi0/0",                             "Saves tokens but may reduce AI accuracy"),
    ("Full subnet mask",     "255.255.255.0",                         None),
    ("CIDR notation",        "/24",                                   "Saves tokens, usually equally clear to AI"),
    ("Full IP + mask",       "ip address 192.168.1.1 255.255.255.0",  None),
    ("CIDR on one line",     "ip address 192.168.1.1/24",             "Not valid IOS but AI understands it"),
    ("Long comment",         "! This is a long comment about the interface configuration purpose", None),
    ("Short comment",        "! uplink to ISP",                       "Saves tokens, same info"),
]

print(f"{'Strategy':<35} {'Tokens':>8}  Notes")
print("-" * 80)

for label, text, note in strategies:
    t = len(enc.encode(text))
    note_str = note or ""
    print(f"{label:<35} {t:>8}  {note_str}")

print()
print("Token optimisation trade-off:")
print("  Abbreviated commands save tokens but may reduce AI accuracy.")
print("  Only abbreviate for bulk extraction tasks with a simple model.")
print("  For security reviews or troubleshooting - use full syntax.")


---
## Demo 5 - Multi-Model Cost Calculator

Now we have the token knowledge - let's build a proper cost calculator
that compares ALL major models side by side.

> **Use this before any batch operation.**
> The same habit as checking bandwidth utilisation before a large file transfer.


In [ ]:
# Model pricing table (2025-2026) - all costs USD per 1 million tokens
MODELS = {
    "claude-haiku-4-5":  {"input":  0.80, "output":  4.00, "context":   200_000},
    "claude-sonnet-4-5": {"input":  3.00, "output": 15.00, "context":   200_000},
    "claude-opus-4":     {"input": 15.00, "output": 75.00, "context":   200_000},
    "gpt-4o-mini":       {"input":  0.15, "output":  0.60, "context":   128_000},
    "gpt-4o":            {"input":  2.50, "output": 10.00, "context":   128_000},
    "gemini-1.5-pro":    {"input":  1.25, "output":  5.00, "context": 2_000_000},
}

def cost_for_tokens(input_tokens, output_tokens, model_key):
    m = MODELS[model_key]
    cost = input_tokens * m["input"] / 1_000_000 + output_tokens * m["output"] / 1_000_000
    fits = (input_tokens + output_tokens) <= int(m["context"] * 0.9)
    return cost, fits

def full_cost_report(config_text, expected_output_tokens=2000, label="Config"):
    input_tokens = len(enc.encode(config_text))
    print(f"\n{'='*70}")
    print(f"COST ANALYSIS: {label}")
    print(f"  Characters     : {len(config_text):,}")
    print(f"  Input tokens   : {input_tokens:,}")
    print(f"  Expected output: ~{expected_output_tokens:,} tokens")
    print(f"  Chars/token    : {len(config_text)/input_tokens:.2f}")
    print(f"{'='*70}")
    print(f"{'Model':<22} {'Fits?':>8}  {'Per query':>10}  {'x100/mo':>10}  {'x1000/mo':>11}")
    print("-" * 70)
    for model in MODELS:
        cost, fits = cost_for_tokens(input_tokens, expected_output_tokens, model)
        status = "YES" if fits else "OVERFLOW"
        print(f"{model:<22} {status:>8}  ${cost:>9.4f}  ${cost*100:>9.2f}  ${cost*1000:>10.2f}")
    print()
    cheapest = min((m for m in MODELS if cost_for_tokens(input_tokens, expected_output_tokens, m)[1]),
        key=lambda m: cost_for_tokens(input_tokens, expected_output_tokens, m)[0], default=None)
    if cheapest:
        c, _ = cost_for_tokens(input_tokens, expected_output_tokens, cheapest)
        print(f"  Cheapest model that fits: {cheapest} (${c:.4f}/query)")

full_cost_report(sample_config,      expected_output_tokens=1500, label="Small switch config (~500 lines)")
full_cost_report(sample_config * 10, expected_output_tokens=3000, label="Medium router config (~5,000 lines)")
full_cost_report(sample_config * 60, expected_output_tokens=5000, label="Large core router config (~30,000 lines)")


---
## Demo 6 - tiktoken Estimate vs Claude API Exact Count

**tiktoken** is fast and free but approximate - it uses OpenAIs tokeniser,
not Claudes. The Claude API has an exact count_tokens endpoint.

When to use each:
- **tiktoken** - quick local estimates, no API cost
- **Claude API count_tokens** - exact count before expensive batch jobs

Rule of thumb: tiktoken is typically within **5-10%** of Claudes actual count.


In [ ]:
test_texts = [
    ("Plain English sentence", "The quick brown fox jumps over the lazy network engineer."),
    ("IOS interface block", "interface GigabitEthernet0/0\n ip address 192.168.1.1 255.255.255.0\n no shutdown"),
    ("BGP configuration", "router bgp 65001\n neighbor 203.0.113.2 remote-as 1234\n neighbor 203.0.113.2 soft-reconfiguration inbound"),
    ("IP prefix list", "10.0.0.0/8 10.1.0.0/16 172.16.0.0/12 192.168.0.0/16 203.0.113.0/24"),
    ("Full sample config", sample_config),
]

print(f"{'Text type':<28} {'tiktoken':>10} {'Claude API':>12} {'Diff %':>8}  Match?")
print("-" * 68)

for label, text in test_texts:
    tiktoken_count = len(enc.encode(text))
    claude_response = client.messages.count_tokens(
        model=MODEL,
        messages=[{"role": "user", "content": text}]
    )
    claude_count = claude_response.input_tokens
    diff_pct = abs(claude_count - tiktoken_count) / claude_count * 100
    match = "Close (< 10%)" if diff_pct < 10 else "Check (> 10%)"
    print(f"{label:<28} {tiktoken_count:>10,} {claude_count:>12,} {diff_pct:>7.1f}%  {match}")

print()
print("Production rule: use tiktoken for quick planning, Claude API for exact billing estimates.")
print("Always add a 10% buffer to your cost projections when using tiktoken.")


---
## Demo 7 - The Cascade Pattern: Smart Cost Optimisation

The insight: **80% of configs are simple** (clean syntax, no critical issues).
You do not need an expensive model for those.

**The cascade pattern:**
1. Run cheap model (Haiku) - triage every config
2. If Haiku says simple + confident -> done, save money
3. If Haiku finds issues or is unsure -> escalate to Sonnet

In production this saves **60-70% on costs** with no loss in quality where it counts.

> **Networking analogy:** BGP local preference routing.
> Default traffic via cheap path. Critical/complex traffic via premium path.


In [ ]:
def analyze_cascade(config: str, verbose=True) -> dict:
    """Stage 1: Haiku triage. Stage 2: Sonnet deep analysis only if needed."""
    prompt = (
        "Quickly triage this router config. Return JSON with these exact fields:\n"
        "{\"complexity\": \"low\" or \"medium\" or \"high\",\n"
        " \"has_critical_issues\": true or false,\n"
        " \"confidence\": 0.0 to 1.0,\n"
        " \"quick_findings\": [\"finding1\", \"finding2\"]}\n\n"
        f"Config (first 3000 chars):\n{config[:3000]}"
    )
    stage1 = client.messages.create(
        model="claude-haiku-4-5-20251001", max_tokens=300, temperature=0,
        system="You are a network security auditor. Respond ONLY with valid JSON. No markdown.",
        messages=[{"role": "user", "content": prompt}]
    )
    s1_cost = stage1.usage.input_tokens * 0.8e-6 + stage1.usage.output_tokens * 4e-6
    try:
        triage = json.loads(stage1.content[0].text)
    except json.JSONDecodeError:
        triage = {"complexity": "high", "has_critical_issues": True, "confidence": 0.0, "quick_findings": ["JSON parse failed"]}

    if verbose:
        print(f"Stage 1 (Haiku): ${s1_cost:.5f}")
        print(f"  Complexity : {triage.get('complexity')}")
        print(f"  Critical?  : {triage.get('has_critical_issues')}")
        print(f"  Confidence : {triage.get('confidence', 0):.0%}")
        print(f"  Findings   : {triage.get('quick_findings', [])}")

    escalate = (
        triage.get("confidence", 0) < 0.85 or
        triage.get("has_critical_issues", False) or
        triage.get("complexity") == "high"
    )
    if not escalate:
        if verbose:
            print(f"\n-> Haiku is sufficient. Total cost: ${s1_cost:.5f}")
        return {"model_used": "haiku", "result": triage, "total_cost": s1_cost}

    if verbose:
        print("\n-> Escalating to Sonnet...")

    review_prompt = (
        "Perform a comprehensive security review of this router config.\n"
        "Structure your response as:\nCRITICAL ISSUES (if any):\n"
        "BEST PRACTICE VIOLATIONS:\nPOSITIVE FINDINGS:\nTOP REMEDIATION STEPS:\n\nConfig:\n" + config
    )
    stage2 = client.messages.create(
        model="claude-sonnet-4-20250514", max_tokens=1500,
        system="You are a senior network security engineer. Be thorough and specific.",
        messages=[{"role": "user", "content": review_prompt}]
    )
    s2_cost = stage2.usage.input_tokens * 3e-6 + stage2.usage.output_tokens * 15e-6
    total_cost = s1_cost + s2_cost
    if verbose:
        print(f"Stage 2 (Sonnet): ${s2_cost:.5f}")
        print(f"Total cascade cost: ${total_cost:.5f}")
        print(f"\n{stage2.content[0].text[:800]}...")
    return {"model_used": "sonnet", "result": stage2.content[0].text, "total_cost": total_cost}

print("=" * 55)
print("RUNNING CASCADE ANALYSIS")
print("=" * 55)
result = analyze_cascade(sample_config)
print(f"\nFinal: used {result['model_used']}, cost ${result['total_cost']:.5f}")


In [ ]:
import random
random.seed(42)

print("Simulating 10 configs: cascade vs always-Sonnet")
print("=" * 55)

cascade_total, sonnet_total, haiku_count = 0, 0, 0

for i in range(10):
    is_complex  = random.random() < 0.3
    haiku_cost  = 0.00018
    sonnet_cost = 0.00240
    cost = haiku_cost + sonnet_cost if is_complex else haiku_cost
    if not is_complex: haiku_count += 1
    cascade_total += cost
    sonnet_total  += sonnet_cost
    marker = "[COMPLEX]" if is_complex else "[simple] "
    print(f"  Config {i+1:2d}: {marker}  cascade=${cost:.5f}  always-sonnet=${sonnet_cost:.5f}")

savings = (sonnet_total - cascade_total) / sonnet_total * 100
print()
print(f"Total - Cascade pattern:  ${cascade_total:.4f}")
print(f"Total - Always Sonnet:    ${sonnet_total:.4f}")
print(f"Savings:                  ${sonnet_total - cascade_total:.4f}  ({savings:.0f}%)")
print(f"Configs handled by Haiku: {haiku_count}/10")


---
## Demo 8 - Fleet Cost Projection Tool

This is the tool that answers the accounting email before you send it.
**Run this BEFORE any batch AI operation.**

Input your fleet details -> get exact cost projections across models -> choose wisely.

> **The habit:** No batch AI operation without a cost estimate first.
> Same as checking bandwidth utilisation before pushing a large file transfer.


In [ ]:
def fleet_projection(fleet_size, avg_config_lines, expected_output_tokens,
                     operations_per_month, use_cascade=True, cascade_escalation_rate=0.25):
    chars_per_config    = avg_config_lines * 50
    input_tokens        = int(chars_per_config / 3.2)
    total_queries_month = fleet_size * operations_per_month

    print(f"\n{'='*65}")
    print(f"FLEET COST PROJECTION")
    print(f"{'='*65}")
    print(f"  Fleet size             : {fleet_size:,} devices")
    print(f"  Avg config size        : {avg_config_lines:,} lines (~{chars_per_config:,} chars)")
    print(f"  Est. input tokens/query: {input_tokens:,}")
    print(f"  Expected output tokens : {expected_output_tokens:,}")
    print(f"  Operations/month       : {operations_per_month}x per device")
    print(f"  Total monthly queries  : {total_queries_month:,}")
    print()
    print(f"{'Model':<22} {'Fits?':>8}  {'Per query':>10}  {'Monthly':>10}  {'Annual':>10}")
    print("-" * 65)
    results = {}
    for model in MODELS:
        cpq, fits = cost_for_tokens(input_tokens, expected_output_tokens, model)
        monthly_cost = cpq * total_queries_month
        annual_cost  = monthly_cost * 12
        status = "YES" if fits else "OVERFLOW"
        print(f"{model:<22} {status:>8}  ${cpq:>9.4f}  ${monthly_cost:>9.2f}  ${annual_cost:>9.2f}")
        results[model] = {"per_query": cpq, "monthly": monthly_cost, "annual": annual_cost, "fits": fits}

    if use_cascade:
        h_cost, _ = cost_for_tokens(input_tokens, expected_output_tokens, "claude-haiku-4-5")
        s_cost, _ = cost_for_tokens(input_tokens, expected_output_tokens, "claude-sonnet-4-5")
        simple_q   = total_queries_month * (1 - cascade_escalation_rate)
        complex_q  = total_queries_month * cascade_escalation_rate
        c_monthly  = simple_q * h_cost + complex_q * (h_cost + s_cost)
        c_annual   = c_monthly * 12
        s_monthly  = results["claude-sonnet-4-5"]["monthly"]
        savings_pct = (s_monthly - c_monthly) / s_monthly * 100
        print()
        print(f"Cascade (Haiku->Sonnet, {cascade_escalation_rate:.0%} escalation):")
        print(f"   Monthly: ${c_monthly:>9.2f}  Annual: ${c_annual:>9.2f}")
        print(f"   vs always-Sonnet: saves ${s_monthly - c_monthly:.2f}/month ({savings_pct:.0f}%)")

    print()
    valid = {m: r for m, r in results.items() if r["fits"]}
    if valid:
        cheapest = min(valid, key=lambda m: valid[m]["per_query"])
        print(f"Recommendation: {cheapest} (cheapest that fits every config)")
    else:
        print("All models overflow - chunk configs or use Gemini 1.5 Pro (2M context)")

# Scenario 1: Weekly compliance audit, 200 devices
fleet_projection(fleet_size=200,  avg_config_lines=500,  expected_output_tokens=2000,
                 operations_per_month=4,  use_cascade=True, cascade_escalation_rate=0.2)

# Scenario 2: Daily log analysis, 50 core routers
fleet_projection(fleet_size=50,   avg_config_lines=5000, expected_output_tokens=3000,
                 operations_per_month=30, use_cascade=True, cascade_escalation_rate=0.3)

# Scenario 3: One-time config review, 1000 switches
fleet_projection(fleet_size=1000, avg_config_lines=200,  expected_output_tokens=1000,
                 operations_per_month=1,  use_cascade=False)


In [ ]:
print("THE ACCOUNTING EMAIL MYSTERY - SOLVED")
print("=" * 55)

# Tuesday: 20 large router configs, ~8000 lines each
fleet_projection(fleet_size=20, avg_config_lines=8000, expected_output_tokens=4000,
                 operations_per_month=1, use_cascade=False)

# Wednesday: 50 small switch configs, ~800 lines each
fleet_projection(fleet_size=50, avg_config_lines=800,  expected_output_tokens=1500,
                 operations_per_month=1, use_cascade=False)

print()
print("Explanation:")
print("  Tuesday   : fewer configs but MASSIVE (8000 lines) -> high token count -> $47")
print("  Wednesday : more configs but SMALL (800 lines)     -> low token count  -> $8")
print("  Lesson    : cost scales with TOKEN COUNT, not config COUNT.")


---
## You Completed Chapter 2\!

| Demo | What you mastered | Key formula |
|------|------------------|------------|
| 1 | Setup tiktoken + Claude | enc = tiktoken.get_encoding("cl100k_base") |
| 2 | BPE from scratch | Common text -> 1 token; rare/technical -> many tokens |
| 3 | Network text tokenisation | Configs ~ 3.2 chars/token (worse than plain English ~ 4.5) |
| 4 | Numbers are expensive | Each IP address costs ~7 tokens |
| 5 | Multi-model cost calculator | cost = (input x input_price + output x output_price) / 1M |
| 6 | tiktoken vs Claude API | tiktoken ~ plus/minus 5-10%; Claude API = exact |
| 7 | Cascade pattern | Haiku first -> escalate only when needed -> 60-70% savings |
| 8 | Fleet projection tool | Plan before you spend - same as checking bandwidth first |

---

### The Cheat Sheet

| AI Concept | Networking Analogy |
|---|---|
| Token | Packet |
| BPE tokenisation | MPLS label assignment |
| Context window (200K) | MTU / buffer size |
| Context overflow | Packet fragmentation |
| Cheap model (Haiku) | Default route - good enough for simple destinations |
| Expensive model (Opus) | Full BGP table - needed for complex routing decisions |
| Cascade pattern | Local preference / policy-based routing |
| Model parameters | Router RAM - more = can hold more knowledge |

---

### What's Next?

**Chapter 3** - Prompt Engineering for Network Engineers:
how to write prompts that consistently get the right answer,
how to structure system prompts, few-shot examples, and chain-of-thought.

*Keep the fleet projection tool handy - you will use it every time you plan a batch AI operation.*
